# 🌿 CropGuard: staged multi-task training (Kaggle GPU)

Trains the model that `app/api` serves, using the real pipeline:
`model/training/train.py` in 5 stages x 10 epochs.

**Attach in the right sidebar:**
| dataset | why |
|---|---|
| `ganrajlongadge/cropguard-corpus-claude` | the 32,897-image corpus |
| `ganrajlongadge/cropguard-train-code` | `model/` (architecture + training code) |

**Accelerator:** GPU T4 x2. `train.py` refuses compute capability < 7.0, so a
P100 will abort on purpose.

This mirrors the pushed kernel `ganrajlongadge/cropguard-v6-train`; push that with
`kaggle kernels push` when you want it to run unattended instead.

In [ ]:
# STEP 1: dependencies. timm provides the FasterNet-T0 backbone.
!pip install -q timm
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('capability', torch.cuda.get_device_capability() if torch.cuda.is_available() else '-')

In [ ]:
# STEP 2: locate the code and the corpus.
#
# A Kaggle dataset created from a single top-level folder is zipped and
# extracted at the mount root, so `model/architecture` can arrive as
# `architecture`. Find train.py wherever it landed instead of assuming.
import glob, os, shutil

IN = '/kaggle/input'
DATA = os.path.join(IN, 'cropguard-corpus-claude')

cands = glob.glob(os.path.join(IN, '**', 'train.py'), recursive=True)
assert cands, f'train.py not found under {IN}; contents={sorted(os.listdir(IN))}'
train_py, pkg = cands[0], os.path.dirname(cands[0])
base = os.path.dirname(pkg)

# NB: do NOT call this 'code' -- /kaggle/working/code shadows the stdlib `code`
# module and breaks nbconvert.
WORK = '/kaggle/working/cgcode'
shutil.rmtree(WORK, ignore_errors=True)
if os.path.basename(base) == 'model':
    shutil.copytree(base, os.path.join(WORK, 'model'))
else:  # flattened mount: wrap architecture/ and training/ back under model/
    os.makedirs(os.path.join(WORK, 'model'), exist_ok=True)
    for sub in ('architecture', 'training'):
        src = os.path.join(base, sub)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(WORK, 'model', sub))

TRAIN = os.path.join(WORK, 'model', 'training', 'train.py')
assert os.path.exists(TRAIN), sorted(os.listdir(os.path.join(WORK, 'model')))
assert os.path.isdir(os.path.join(DATA, 'images')), sorted(os.listdir(DATA))

for split in ('train', 'val', 'test'):
    n = len(glob.glob(os.path.join(DATA, 'images', split, '*.jpg')))
    print(f'{split:6s} {n:6d} images')
print('\ntrain.py ->', TRAIN)

In [ ]:
# STEP 3: train.
#
# 5 stages x 10 epochs, matching the paper's Table 1/2 schedule:
#   1 backbone+neck+det   2 cls head   3 seg head   4 joint   5 +severity
# Measured cost: ~350-400 s/epoch, peak ~6.4 GB VRAM at batch 32, ~5-6 h total.
# Reduce BATCH_SIZE to 16 on a 16 GB card if you hit OOM.
import os, subprocess, sys

env = dict(os.environ)
env['PYTHONPATH'] = WORK
env['CROPGUARD_DATA'] = DATA

cmd = [sys.executable, TRAIN,
       '--root', DATA,
       '--out', '/kaggle/working/runs',
       '--stages', '1,2,3,4,5',
       '--stage-epochs', '10',
       '--batch-size', '32',
       '--size', '320',
       '--workers', '2',
       '--amp',
       '--log-every', '50']
print('RUN:', ' '.join(cmd), flush=True)
rc = subprocess.call(cmd, env=env)
print('train.py exited with', rc)

## ✅ After training

Checkpoints land in `/kaggle/working/runs`:

```
runs/student_best.pt              <- the model to serve
runs/checkpoints/best_stage{1..5}.pt, last.pt   <- resumable
runs/logs/stage{1..5}.log                        <- per-epoch losses
```

Download them (from a terminal, not the notebook):

```bash
kaggle kernels output ganrajlongadge/cropguard-v6-train -p ./out
```

### Scoring on the test split

`app/api/eval.py` produces `metrics.json` (mAP, cls_top1, seg IoU/Dice, severity
MAE/RMSE/R2/QWK). It imports its siblings flat (`from dataset import ...`,
`from model import ...`, `from losses import ...`), so it needs those modules
next to it -- add `eval.py`, `dataset.py`, `losses.py`, `model.py`,
`inference_utils.py` from `app/api/` to the code dataset to run it here.

### Two honest caveats about what these numbers mean

* this corpus's masks are **leaf** regions, so `seg_mIoU` is leaf-region IoU, not
  lesion segmentation;
* severity labels derive from leaf coverage, so the severity head scores worse
  than a constant predictor (`sev_r2 -1.41`) and the API flags it
  `severity.reliable = false`.

Fixing either needs different labels, not more epochs -- see `HANDOFF.md` and
`data_pipeline/scripts/prepare_plantseg.py`.